# Query Decomposition [Step 3 - Breaking Complex Queries into Sub-Queries]

> **MLCourse - Agentic AI - Agentic RAG**

Complex questions often cannot be answered with a single retrieval. Query
decomposition breaks a multi-part question into focused sub-queries, each
targeting a specific aspect. This notebook builds a chain that decomposes
a complex query, retrieves for each sub-query, and synthesizes a combined
answer.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

False

In [2]:
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] API key found")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")

[GREEN] No API key needed -- using local ChatOllama


In [3]:
# ## 1. Load and Chunk the Document

from langchain_text_splitters import CharacterTextSplitter

text_path = r"D:\projects\python\MLCourse\03_agentic_ai\data\alice.txt"
with open(text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_text(raw_text)
print(f"Loaded alice.txt: {len(raw_text)} chars -> {len(chunks)} chunks")

Loaded alice.txt: 144696 chars -> 191 chunks


In [4]:
# ## 2. Build the Vector Store

from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_texts(chunks, embeddings, collection_name="alice_decomp")
print(f"Vector store built with {vectorstore._collection.count()} vectors")

Vector store built with 191 vectors


In [5]:
# ## 3. Initialize the LLM

from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:8b", temperature=0)
print("LLM initialized:", llm.model)

LLM initialized: llama3.1:8b


In [6]:
# ## 4. Build the Decomposition Chain
# The decomposition prompt asks the LLM to break a complex query into
# focused sub-queries. Each sub-query targets one specific aspect.

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

decompose_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a query decomposition expert. Break complex questions into "
     "simpler, self-contained sub-queries that can each be answered independently.\n\n"
     "RULES:\n"
     "- Each sub-query must be specific and self-contained\n"
     "- Preserve the original intent of each part\n"
     "- Order sub-queries logically (general first, then specific)\n"
     "- Do NOT repeat sub-queries that are nearly identical\n"
     "- Return ONLY the sub-queries, one per line, numbered 1, 2, 3, etc."),
    ("user", "{query}")
])

decompose_chain = decompose_prompt | llm | StrOutputParser()
print("Decomposition chain built.")

Decomposition chain built.


In [7]:
# ## 5. Test Decomposition on a Complex Query

complex_query = (
    "What are the main themes in Alice in Wonderland "
    "and how do they relate to Victorian society?"
)

print("=" * 60)
print(f"Complex query: {complex_query}")
print("=" * 60)

result = decompose_chain.invoke({"query": complex_query})
print("\nDecomposed sub-queries:")
print(result)

Complex query: What are the main themes in Alice in Wonderland and how do they relate to Victorian society?



Decomposed sub-queries:
Here are the sub-queries:

1. What are the main themes present in the novel "Alice's Adventures in Wonderland" by Lewis Carroll?
2. How did Victorian society view children and their role in society during the 19th century?
3. In what ways does the character of Alice reflect or challenge societal expectations of women during the Victorian era?
4. What commentary on class structure and social hierarchy can be found in the novel's depiction of Wonderland's inhabitants?
5. How do the themes of identity, morality, and logic relate to the intellectual and philosophical debates of the time?


In [8]:
# ## 6. Parse Sub-Queries into a List

import re

def parse_sub_queries(text: str) -> list:
    """Extract numbered sub-queries from LLM output."""
    lines = text.strip().split("\n")
    queries = []
    for line in lines:
        # Remove numbering: "1. ", "1) ", etc.
        cleaned = re.sub(r"^\d+[\.\)]\s*", "", line.strip())
        if cleaned and len(cleaned) > 5:
            queries.append(cleaned)
    return queries

sub_queries = parse_sub_queries(result)
print(f"Parsed {len(sub_queries)} sub-queries:")
for i, sq in enumerate(sub_queries, 1):
    print(f"  {i}. {sq}")

Parsed 6 sub-queries:
  1. Here are the sub-queries:
  2. What are the main themes present in the novel "Alice's Adventures in Wonderland" by Lewis Carroll?
  3. How did Victorian society view children and their role in society during the 19th century?
  4. In what ways does the character of Alice reflect or challenge societal expectations of women during the Victorian era?
  5. What commentary on class structure and social hierarchy can be found in the novel's depiction of Wonderland's inhabitants?
  6. How do the themes of identity, morality, and logic relate to the intellectual and philosophical debates of the time?


In [9]:
# ## 7. Build the Retrieval Function
# Simple similarity search for each sub-query.

def retrieve_for_query(query: str, k: int = 3) -> str:
    """Retrieve relevant documents for a single query."""
    docs = vectorstore.similarity_search(query, k=k)
    context = "\n\n".join([d.page_content for d in docs])
    return context

In [10]:
# ## 8. Build the Sub-Query Answering Chain
# For each sub-query, retrieve and generate a focused answer.

sub_answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using ONLY the provided context. "
     "Be specific and concise. If the context is insufficient, say so."),
    ("user", "Context:\n{context}\n\nQuestion: {query}")
])

sub_answer_chain = sub_answer_prompt | llm | StrOutputParser()
print("Sub-answer chain built.")

Sub-answer chain built.


In [11]:
# ## 9. Build the Synthesis Chain
# Combines all sub-answers into a coherent response.

synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a synthesis expert. Combine the following sub-answers into a "
     "single, coherent response to the original complex question.\n\n"
     "RULES:\n"
     "- Maintain logical flow between topics\n"
     "- Remove redundancy across sub-answers\n"
     "- Ensure the final answer directly addresses the original question\n"
     "- Be comprehensive but concise"),
    ("user",
     "Original question: {original_query}\n\n"
     "Sub-answers:\n{sub_answers}")
])

synthesis_chain = synthesis_prompt | llm | StrOutputParser()
print("Synthesis chain built.")

Synthesis chain built.


In [12]:
# ## 10. Build the Full Decomposition Pipeline

def decompose_and_answer(query: str) -> dict:
    """Full pipeline: decompose -> retrieve -> answer each -> synthesize."""
    print(f"\n{'='*60}")
    print(f"Processing: {query}")
    print(f"{'='*60}")

    # Step 1: Decompose
    print("\n[STEP 1] Decomposing query...")
    raw_decomp = decompose_chain.invoke({"query": query})
    sub_queries = parse_sub_queries(raw_decomp)
    print(f"  Got {len(sub_queries)} sub-queries")

    # Step 2: Retrieve and answer each sub-query
    print("\n[STEP 2] Retrieving and answering sub-queries...")
    sub_answers = []
    for i, sq in enumerate(sub_queries, 1):
        print(f"\n  Sub-query {i}: {sq[:80]}...")
        context = retrieve_for_query(sq)
        answer = sub_answer_chain.invoke({"context": context, "query": sq})
        sub_answers.append(f"Sub-query {i}: {sq}\nAnswer: {answer}")
        print(f"  Answer {i}: {answer[:100]}...")

    # Step 3: Synthesize
    print("\n[STEP 3] Synthesizing final answer...")
    combined = "\n\n".join(sub_answers)
    final = synthesis_chain.invoke({
        "original_query": query,
        "sub_answers": combined
    })
    print(f"  Final answer: {final[:150]}...")

    return {
        "original_query": query,
        "sub_queries": sub_queries,
        "sub_answers": sub_answers,
        "final_answer": final
    }

In [13]:
# ## 11. Run the Full Pipeline

result = decompose_and_answer(
    "What are the main themes in Alice in Wonderland "
    "and how do they relate to Victorian society?"
)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["final_answer"])


Processing: What are the main themes in Alice in Wonderland and how do they relate to Victorian society?

[STEP 1] Decomposing query...


  Got 6 sub-queries

[STEP 2] Retrieving and answering sub-queries...

  Sub-query 1: Here are the sub-queries:...


  Answer 1: I'm ready to answer. What are the sub-queries?...

  Sub-query 2: What are the main themes present in the novel "Alice's Adventures in Wonderland"...


  Answer 2: The provided context is insufficient to determine the main themes present in the novel. The text onl...

  Sub-query 3: How did Victorian society view children and their role in society during the 19t...


  Answer 3: The provided context does not directly address how Victorian society viewed children and their role ...

  Sub-query 4: In what ways does the character of Alice reflect or challenge societal expectati...


  Answer 4: The context is insufficient to provide a comprehensive answer. However, some observations can be mad...

  Sub-query 5: What commentary on class structure and social hierarchy can be found in the nove...


  Answer 5: The context is insufficient to provide a specific answer. The provided text only describes the absur...

  Sub-query 6: How do the themes of identity, morality, and logic relate to the intellectual an...


  Answer 6: The context is insufficient to answer this question. The provided text appears to be an excerpt from...

[STEP 3] Synthesizing final answer...


  Final answer: The main themes present in Alice in Wonderland, as written by Lewis Carroll, are closely tied to the societal context of Victorian England during the ...

FINAL ANSWER
The main themes present in Alice in Wonderland, as written by Lewis Carroll, are closely tied to the societal context of Victorian England during the 19th century. One of the primary themes is the exploration of identity, morality, and logic, which were all significant intellectual and philosophical debates of the time.

In terms of class structure and social hierarchy, the novel offers a commentary on the absurdities and illogicalities of Victorian society's rigid social stratification. The inhabitants of Wonderland, such as the March Hare, Hatter, and Dormouse, embody the contradictions and paradoxes of Victorian social norms, highlighting the artificiality and arbitrariness of class distinctions.

The character of Alice herself reflects and challenges societal expectations of women during the Victorian

In [14]:
# ## 12. Test with a Different Complex Query

result2 = decompose_and_answer(
    "Compare how Alice treats animals versus how humans treat her, "
    "and identify what this says about the book's message."
)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result2["final_answer"])


Processing: Compare how Alice treats animals versus how humans treat her, and identify what this says about the book's message.

[STEP 1] Decomposing query...


  Got 6 sub-queries

[STEP 2] Retrieving and answering sub-queries...

  Sub-query 1: Here are the decomposed sub-queries:...


  Answer 1: There are no decomposed sub-queries provided in the context. The text appears to be a passage from L...

  Sub-query 2: What is the nature of Alice's interactions with animals in the story?...


  Answer 2: Alice has a friendly interaction with a Mouse, but it quickly turns into a negative one when she tri...

  Sub-query 3: How do humans interact with Alice in the story?...


  Answer 3: In the provided context, humans (specifically, the Queen and the King) interact with Alice by:

* Qu...

  Sub-query 4: What specific behaviors or actions does Alice exhibit towards animals that can b...


  Answer 4: Alice exhibits the following behaviors or actions towards animals:

1. She tries to show the Mouse a...

  Sub-query 5: What message about human relationships and treatment of others does this compari...


  Answer 5: The comparison between "flamingoes and mustard" conveys that people who are similar in nature or beh...

  Sub-query 6: Is there a broader theme or commentary on societal norms or expectations in the ...


  Answer 6: Yes, there is a broader theme and commentary on societal norms and expectations in this passage. The...

[STEP 3] Synthesizing final answer...


  Final answer: Here is a comprehensive answer that combines the sub-answers into a single, coherent response:

In Lewis Carroll's "Alice's Adventures in Wonderland,"...

FINAL ANSWER
Here is a comprehensive answer that combines the sub-answers into a single, coherent response:

In Lewis Carroll's "Alice's Adventures in Wonderland," Alice's interactions with animals and humans reveal a commentary on societal norms and expectations. When comparing how Alice treats animals versus how humans treat her, it becomes clear that both involve attempts to control or manipulate others for one's own purposes.

Alice's interactions with animals are limited but telling. She tries to show the Mouse a dog, seemingly eager to distract it from its swimming away, and when she thinks about taking care of the creature (which turns out to be a pig), she considers carrying it home without much concern for its well-being. These behaviors can be compared to how humans treat her in that they both involve Alice 

In [15]:
# ## 13. Test with a Three-Part Query

result3 = decompose_and_answer(
    "Describe the Mad Hatter's tea party, explain why it is considered "
    "nonsensical, and compare it to real Victorian social customs."
)

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result3["final_answer"])


Processing: Describe the Mad Hatter's tea party, explain why it is considered nonsensical, and compare it to real Victorian social customs.

[STEP 1] Decomposing query...


  Got 8 sub-queries

[STEP 2] Retrieving and answering sub-queries...

  Sub-query 1: Here are the sub-queries:...


  Answer 1: I'm ready to answer. What are the sub-queries?...

  Sub-query 2: What was the setting of the Mad Hatter's tea party in Lewis Carroll's "Alice's A...


  Answer 2: The setting of the Mad Hatter's tea party in Lewis Carroll's "Alice's Adventures in Wonderland" was ...

  Sub-query 3: Who attended the Mad Hatter's tea party in Lewis Carroll's "Alice's Adventures i...


  Answer 3: The March Hare and the Hatter were having tea together. A Dormouse was also present, but he was fast...

  Sub-query 4: What were some notable events or conversations that occurred during the Mad Hatt...


...

  Answer 5: The provided context does not explicitly state why the Mad Hatter's tea party is considered nonsensi...

  Sub-query 6: In what ways does the Mad Hatter's tea party subvert traditional Victorian socia...


  Answer 6: The Mad Hatter's tea party subverts traditional Victorian social customs in several ways:

1. Unconv...

  Sub-query 7: What were some common features of real Victorian-era tea parties, such as those ...


  Answer 7: The provided context does not describe a traditional Victorian-era tea party. In fact, it describes ...

  Sub-query 8: How did the Mad Hatter's tea party differ from or reflect the social norms and e...


  Answer 8: The context is insufficient to answer this question as it only provides a passage from "Alice's Adve...

[STEP 3] Synthesizing final answer...


  Final answer: Here's a comprehensive answer to the original question:

The Mad Hatter's tea party, as depicted in Lewis Carroll's "Alice's Adventures in Wonderland,...

FINAL ANSWER
Here's a comprehensive answer to the original question:

The Mad Hatter's tea party, as depicted in Lewis Carroll's "Alice's Adventures in Wonderland," is a fantastical and absurd scene that takes place under a tree in front of the March Hare's house. The party consists of the March Hare, the Hatter, and a Dormouse, who is fast asleep between them. Alice later joins them at the table, where they engage in illogical conversations and behaviors.

The Mad Hatter's tea party is considered nonsensical by literary critics and readers because it subverts traditional Victorian social customs surrounding tea parties. In real Victorian-era tea parties, which were typically hosted by the upper class in England during the 19th century, there was a strong emphasis on formality, etiquette, and propriety. Tea was served

In [16]:
# ## 14. Visualize the Pipeline as a Graph

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class DecompState(TypedDict):
    query: str
    sub_queries: list
    sub_answers: list
    final_answer: str

print("=== Decomposition Pipeline Structure ===")
print("  START -> decompose")
print("           |")
print("           v")
print("  For each sub-query:")
print("    retrieve -> sub_answer")
print("           |")
print("           v")
print("  synthesize -> END")
print()
print("This is a sequential pipeline, not a branching graph.")
print("Each sub-query is handled independently, then combined.")

=== Decomposition Pipeline Structure ===
  START -> decompose
           |
           v
  For each sub-query:
    retrieve -> sub_answer
           |
           v
  synthesize -> END

This is a sequential pipeline, not a branching graph.
Each sub-query is handled independently, then combined.


In [17]:
# ## Summary
#
# Key takeaways:
# - Query decomposition splits complex questions into focused sub-queries
# - Each sub-query targets one specific aspect of the original question
# - Retrieval and answering happen independently per sub-query
# - Synthesis combines sub-answers into a coherent final response
# - This pattern works well for multi-part, comparative, or analytical queries
# - The decomposition step is the key innovation: it makes the problem tractable
# - Sub-queries can be processed in parallel (shown here sequentially for clarity)